# 🌐 Diplomatic & Propaganda Language Analysis with NLP

**Goal:** measure the *hostility*, *cooperation* and *rhetorical shifts* of countries year by year, using speeches by heads of state and foreign ministers at the **UN General Assembly General Debate**, or a collection of texts you gather yourself (joint statements, state-media articles…).

**Example research question:** How has the rhetoric of Russia, the United States, China and Vietnam at the UN changed since 2000? Do the "turning points" coincide with major events (Iraq 2003, Crimea 2014, Ukraine 2022)?

### Pipeline

```
Step 2: Raw texts (.txt/.csv) → a data table
Step 3: Cleaning → sentence splitting
Steps 4–6: Scoring with 3 layers of measurement
Step 7: Aggregate by year → charts
Step 8: Find turning points → close-read the sentences to validate
```

**Three layers of measurement** (from simple to advanced):
1. **VADER** – a general-purpose sentiment tool (is this sentence positive or negative?).
2. **Diplomatic dictionaries** – your own lists of "hostile" vs "cooperative" words, closer to the language of International Relations.
3. **(Advanced) Zero-shot AI model** – a language model reads each sentence and classifies its tone.

### 🧭 How to run this notebook (for non-coders)
1. Open it in **Google Colab**: go to https://colab.research.google.com → *File → Upload notebook* → pick this `.ipynb` file.
2. Each grey box is a code cell. Click a cell and press **Shift + Enter** to run it. Run them **in order, top to bottom**.
3. You **only need to edit the "Step 1 – Settings" cell**. Run all other cells as they are.
4. Got an error? Read the **last line** of the error message (it usually says what went wrong), take a screenshot and ask for help.

## Step 0 – Install libraries

A *library* is a toolkit someone else wrote that we can reuse. Colab already has `pandas` (data tables, like Excel) and `matplotlib` (charts). We only need to add `vaderSentiment` (the sentiment tool).

In [ ]:
%pip install -q vaderSentiment

In [ ]:
import os, re, glob, tarfile, zipfile, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import requests
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 150)
print("✅ Libraries loaded")

## Step 1 – Research settings ✏️ *(the only cell you need to edit)*

- **Country codes** use the 3-letter ISO standard: `RUS` Russia, `USA` United States, `CHN` China, `VNM` Vietnam, `GBR` United Kingdom, `FRA` France, `IND` India, `UKR` Ukraine, `PRK` North Korea, `IRN` Iran, `ISR` Israel…
- Pick **2–4 countries** so the charts stay readable (maximum 8).
- `KEYWORDS`: words or phrases whose frequency you want to track over the years. Use `*` to catch every ending, e.g. `sanction*` matches *sanction, sanctions, sanctioned*.
- `EVENTS`: years drawn as dotted lines on the charts so you can compare.

In [ ]:
# Data source: "UNGDC" = UN General Assembly speeches (downloaded automatically)
#              "CSV"   = your own data (joint statements, articles...)
DATA_SOURCE = "UNGDC"
MY_CSV_FILE = "my_data.csv"   # only used when DATA_SOURCE = "CSV"

# Countries to compare
COUNTRIES = ["RUS", "USA", "CHN", "VNM"]

# Time range (a large END_YEAR = up to the latest year in the data)
START_YEAR = 2000
END_YEAR = 2100

# Keywords to track (case-insensitive; * = any word ending)
KEYWORDS = ["NATO", "sovereign*", "multipolar*", "sanction*", "hegemon*",
            "terroris*", "climate", "Ukrain*", "Taiwan", "South China Sea"]

# Events to mark on the charts
EVENTS = {2001: "9/11", 2003: "Iraq War", 2008: "Financial crisis",
          2014: "Crimea", 2020: "COVID-19", 2022: "Russia–Ukraine war"}

# Compare with the average of all countries in the data?
COMPARE_TO_AVERAGE = True

## Step 2 – Prepare the data

### 2.1 About the UNGDC dataset
- The **United Nations General Debate Corpus**, built by Slava Jankin, Alexander Baturo and Niheer Dasandi, is publicly available on **Harvard Dataverse**: https://doi.org/10.7910/DVN/0TJX8Y
- It holds 10,000+ speeches by ~200 countries from 1946 to today. Each `.txt` file is **one country's speech in one year**. File names look like `RUS_77_2022.txt` = Russia · session 77 · year 2022.
- The texts are the **official UN English versions**. Many were translated from Russian, Chinese, French, Spanish… ⇒ we are measuring the tone *of the translation*. State this clearly in the methods section of your paper.
- **Citation:** Jankin, S., Baturo, A., & Dasandi, N. (2024). *Words to unite nations: The complete United Nations General Debate Corpus, 1946–present.* Journal of Peace Research.

The code cell below downloads the data automatically through the Dataverse API. **If the automatic download fails**, do it by hand:
1. Open https://doi.org/10.7910/DVN/0TJX8Y → find the archive containing the speeches (ending in `.tgz` or `.zip`) → download it.
2. Run the **"📤 Manual upload"** cell right after and choose the downloaded file.

*(Tip: if the file is large and your connection slow, put it on Google Drive and use the folder icon on Colab's left bar → "Mount Drive".)*

In [ ]:
DATA_DIR = Path("ungdc_raw")
UNGDC_DOI = "doi:10.7910/DVN/0TJX8Y"
DATAVERSE_API = "https://dataverse.harvard.edu/api"


def extract_archive(path, target=DATA_DIR):
    """Extract a .tgz / .tar.gz / .zip file into the target folder."""
    path = str(path)
    Path(target).mkdir(parents=True, exist_ok=True)
    if path.lower().endswith((".tgz", ".tar.gz", ".tar")):
        with tarfile.open(path) as tar:
            try:
                tar.extractall(target, filter="data")   # safe extraction
            except TypeError:                           # older Python versions
                tar.extractall(target)
    elif path.lower().endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            z.extractall(target)
    else:
        raise ValueError(f"Unrecognised file format: {path}")


def download_ungdc():
    """Ask Dataverse for the dataset's file list, then download the largest archive (the speeches)."""
    print("🔎 Asking Harvard Dataverse for the list of files...")
    r = requests.get(f"{DATAVERSE_API}/datasets/:persistentId/",
                     params={"persistentId": UNGDC_DOI}, timeout=60)
    r.raise_for_status()
    version = r.json()["data"]["latestVersion"]
    print(f"   Dataset version: {version.get('versionNumber')}.{version.get('versionMinorNumber')}")
    files = [f["dataFile"] for f in version["files"]]
    for f in files:
        print(f"   - {f['filename']}  ({f.get('filesize', 0) / 1e6:.1f} MB)")

    archives = [f for f in files if f["filename"].lower().endswith((".tgz", ".tar.gz", ".zip"))]
    if not archives:
        raise RuntimeError("No archive (.tgz/.zip) found in the dataset.")
    chosen = max(archives, key=lambda f: f.get("filesize", 0))

    target = Path(chosen["filename"])
    print(f"⬇️  Downloading {target.name} (may take 1–3 minutes)...")
    with requests.get(f"{DATAVERSE_API}/access/datafile/{chosen['id']}", stream=True, timeout=600) as resp:
        resp.raise_for_status()
        downloaded, next_report = 0, 20 << 20
        with open(target, "wb") as out:
            for chunk in resp.iter_content(chunk_size=1 << 20):
                out.write(chunk)
                downloaded += len(chunk)
                if downloaded >= next_report:          # progress every ~20 MB
                    print(f"   ...{downloaded / 1e6:.0f} MB")
                    next_report += 20 << 20
    print("📦 Extracting...")
    extract_archive(target, DATA_DIR)
    print("✅ Download and extraction complete!")


already_have_data = DATA_DIR.exists() and any(DATA_DIR.rglob("*.txt"))
if DATA_SOURCE == "UNGDC" and already_have_data:
    print("✅ Data already present, skipping the download.")
elif DATA_SOURCE == "UNGDC":
    try:
        download_ungdc()
    except Exception as err:
        print("⚠️ Automatic download failed:", err)
        print("👉 Download the file by hand (instructions above), then run the '📤 Manual upload' cell.")
else:
    print("You are using your own CSV → run the '📤 Manual upload' cell to upload it.")

In [ ]:
# 📤 MANUAL UPLOAD
# Only run this cell if (a) the automatic download above failed, or (b) you are using your own CSV.
# A "Choose Files" button will appear below the cell.
try:
    from google.colab import files
    uploaded = files.upload()
    for name in uploaded:
        if name.lower().endswith((".tgz", ".tar.gz", ".zip")):
            extract_archive(name, DATA_DIR)
            print("✅ Extracted", name)
        else:
            print("✅ Uploaded", name)
except ImportError:
    print("You are not on Colab: put the file in the same folder as this notebook.")

### 2.2 Read the text files into a *data table*

We gather all texts into one table (a *DataFrame*, like an Excel sheet): **one row per text**, with columns `country` (code), `year`, `text` (full text)…

**Using your own data (`DATA_SOURCE = "CSV"`):** the CSV needs at least 3 columns `country`, `year`, `text` (see the sample file `sample_data.csv`). A year may have several texts (several articles or statements) – they are added up per year automatically.

⚠️ **Texts must be in English**, because VADER and the dictionaries in this notebook are English. For state media, use the official English editions (e.g. Xinhua English, TASS English, VNA English) or one consistent translation.

In [ ]:
FILENAME_PATTERN = re.compile(r"^([A-Z]{2,4})_(\d{1,3})_(\d{4})\.txt$")


def read_ungdc(folder):
    """Read every .txt file named like XXX_77_2022.txt in the folder (and subfolders)."""
    rows = []
    for p in Path(folder).rglob("*.txt"):
        m = FILENAME_PATTERN.match(p.name)
        if not m:
            continue   # skip files that don't match (e.g. system files like "._ABC")
        country, session, year = m.groups()
        rows.append({"country": country, "session": int(session), "year": int(year),
                     "source": "UN General Debate",
                     "text": p.read_text(encoding="utf-8", errors="replace")})
    return pd.DataFrame(rows)


def read_my_csv(path):
    """Read your CSV; it must have the columns country, year, text."""
    table = pd.read_csv(path)
    table.columns = [c.strip().lower() for c in table.columns]
    missing = {"country", "year", "text"} - set(table.columns)
    if missing:
        raise ValueError(f"The CSV is missing the column(s) {missing}. Required: country, year, text")
    table["country"] = table["country"].astype(str).str.strip().str.upper()
    table["year"] = pd.to_numeric(table["year"], errors="coerce")
    table = table.dropna(subset=["year", "text"])
    table["year"] = table["year"].astype(int)
    if "source" not in table.columns:
        table["source"] = "CSV"
    return table


raw_docs = read_ungdc(DATA_DIR) if DATA_SOURCE == "UNGDC" else read_my_csv(MY_CSV_FILE)
if raw_docs.empty:
    raise RuntimeError("❌ No texts were read. Check Step 2.1 (download/extract the data).")

raw_docs = raw_docs.sort_values(["country", "year"]).reset_index(drop=True)
raw_docs["doc_id"] = range(len(raw_docs))
raw_docs["words"] = raw_docs["text"].str.split().str.len()

print(f"📚 Read {len(raw_docs):,} texts from {raw_docs.country.nunique()} countries, "
      f"{raw_docs.year.min()} to {raw_docs.year.max()}.")

### 2.3 Inspect the data – *always look at the data before analysing it!*

A good data engineer never runs a model without looking at the data first. The cell below:
- filters to the countries and years you chose,
- reports any country that is **missing** from the data (usually a typo in the code),
- prints the first 600 characters of one speech.

In [ ]:
in_range = raw_docs.year.between(START_YEAR, END_YEAR)
world_docs = raw_docs[in_range].copy()            # used for the all-country average
docs = world_docs[world_docs.country.isin(COUNTRIES)].copy()

for code in COUNTRIES:
    n = (docs.country == code).sum()
    if n == 0:
        print(f"⚠️ No texts for '{code}' in the chosen years. Check the country code!")
        similar = [c for c in raw_docs.country.unique() if c[:2] == code[:2]]
        if similar:
            print("   Similar codes in the data:", similar)
    else:
        years = docs.loc[docs.country == code, "year"]
        print(f"✅ {code}: {n} texts ({years.min()}–{years.max()}), "
              f"{docs.loc[docs.country == code, 'words'].mean():.0f} words per text on average")

if docs.empty:
    raise RuntimeError("❌ No texts left after filtering. Edit COUNTRIES or the years in Step 1.")

example = docs.iloc[-1]
print(f"\n📄 Excerpt: {example.country} – {example.year}\n")
print(example.text[:600], "...")

## Step 3 – Cleaning & sentence splitting

**Why split into sentences?** VADER was designed for *short* texts (like a tweet). Fed a whole 3,000-word speech, its score almost always hits the ceiling of +1 or −1 and becomes meaningless. So we score **each sentence**, then average them for the whole speech.

Cleaning means: removing paragraph numbers at the start of lines ("12."), merging extra line breaks, and protecting abbreviations like *Mr.* and *U.S.* so they are not mistaken for the end of a sentence.

In [ ]:
ABBREVIATIONS = re.compile(r"\b(Mr|Mrs|Ms|Dr|St|No|Nos|Art|Arts|Vol|para|paras|U\.S|U\.N|U\.K|E\.U|H\.E)\.")
SENTENCE_BREAK = re.compile(r"(?<=[.!?])\s+(?=[\"“'(\[]?[A-Z])")


def clean_text(text):
    text = text.replace("﻿", " ")
    text = re.sub(r"(?m)^\s*\d{1,3}\.\s+", "", text)   # paragraph numbers at the start of a line
    text = re.sub(r"\s+", " ", text)                   # merge line breaks & extra spaces
    return text.strip()


def split_sentences(text):
    protected = ABBREVIATIONS.sub(lambda m: m.group(0).replace(".", "§"), text)
    parts = [s.replace("§", ".").strip() for s in SENTENCE_BREAK.split(protected)]
    return [s for s in parts if len(s.split()) >= 4]   # drop very short fragments


docs["clean"] = docs["text"].apply(clean_text)
world_docs["clean"] = world_docs["text"].apply(clean_text)

sentences = pd.DataFrame(
    [{"doc_id": d.doc_id, "country": d.country, "year": d.year, "sentence": s}
     for d in docs.itertuples() for s in split_sentences(d.clean)]
)
print(f"✂️ Split {len(sentences):,} sentences from {len(docs)} texts.")
sentences.sample(5, random_state=1)

## Step 4 – Layer 1: sentiment analysis with VADER

VADER gives each sentence a **compound** score from **−1** (very negative) to **+1** (very positive). A common convention: ≤ −0.05 is negative, ≥ +0.05 is positive.

From this we compute, for each country–year:
- `vader_mean`: the average score of all sentences,
- `negative_share`: the share of negative sentences.

⚠️ **Important limitation:** VADER was built for social media, not diplomacy. UN language is formal and full of congratulations ("*I congratulate…*"), so scores tend to **lean positive**. Focus on **trends over time** and **comparisons between countries**, not the absolute numbers.

In [ ]:
vader = SentimentIntensityAnalyzer()
sentences["vader"] = sentences["sentence"].apply(lambda s: vader.polarity_scores(s)["compound"])
sentences["is_negative"] = sentences["vader"] <= -0.05

print("The 5 sentences VADER scored most negative:")
sentences.nsmallest(5, "vader")[["country", "year", "vader", "sentence"]]

## Step 5 – Layer 2: diplomatic dictionaries (*dictionary method*)

This is a classic method in "text as data" research: define in advance the words that signal **hostility** and **cooperation**, then count them. To compare long and short speeches fairly, we count **occurrences per 1,000 words**.

- `hostility` = hostile words per 1,000 words
- `cooperation` = cooperative words per 1,000 words
- `tone_balance` = (cooperative − hostile) / (cooperative + hostile), from **−1** (fully confrontational) to **+1** (fully conciliatory).

✏️ **You should adapt the two lists below** to your research question, and justify your choices in the methods section. For example, research on China might add *"hegemonism"* or *"Cold War mentality"*; research on Russia might add *"Russophob*"* or *"neo-Nazi*"*. `*` = any word ending.

In [ ]:
HOSTILE_TERMS = [
    "aggress*", "hostil*", "threat*", "provocat*", "confrontation*", "belligeren*", "warmonger*",
    "invasion*", "invade*", "occupation", "occupying", "occupier*", "annex*", "interfer*",
    "sanction*", "blockade*", "embargo*", "coerc*", "blackmail*", "intimidat*",
    "hegemon*", "imperialis*", "neo-colonial*", "neocolonial*", "colonialis*", "expansionis*",
    "destabili*", "subversi*", "regime change", "colour revolution*", "color revolution*",
    "double standard*", "hypocri*", "propaganda", "disinformation", "falsif*",
    "condemn*", "denounce*", "violat*", "illegal*", "illegitimate", "unlawful",
    "atrocit*", "genocid*", "massacre*", "terroris*", "extremis*", "nazi*", "neo-nazi*", "fascis*",
    "enemy", "enemies", "attack*", "war", "wars", "bombing*", "cold war", "zero-sum",
    "bloc mentality", "unilateralis*", "arrogan*", "domination", "dictat*",
]

COOPERATIVE_TERMS = [
    "cooperat*", "co-operat*", "collaborat*", "partner*", "dialogue*", "negotiat*", "consultation*",
    "multilateral*", "mutual*", "reciproc*", "friendship", "friendly", "solidarity", "together",
    "trust", "confidence-building", "reconcil*", "consensus", "compromise*", "understanding",
    "peaceful*", "peacebuilding", "win-win", "common interest*", "shared future", "common future",
    "common destiny", "jointly", "integrat*", "connectivity", "assist*", "goodwill",
    "good-neighbo*", "good neighbo*", "harmon*", "inclusive*", "welcome*", "agreement*",
    "build bridges", "bridge-building",
]


def build_regex(terms):
    """Turn a list of terms (with optional *) into one search pattern (regex)."""
    patterns = []
    for term in terms:
        term = term.strip().lower()
        stem = re.escape(term.rstrip("*")).replace(r"\ ", r"\s+")
        patterns.append(stem + (r"[\w-]*" if term.endswith("*") else ""))
    patterns.sort(key=len, reverse=True)      # longer phrases first ("cold war" before "war")
    return re.compile(r"\b(?:" + "|".join(patterns) + r")\b", flags=re.IGNORECASE)


HOSTILE_RX = build_regex(HOSTILE_TERMS)
COOP_RX = build_regex(COOPERATIVE_TERMS)


def count_dictionary_terms(table):
    table = table.copy()
    table["hostile_hits"] = table["clean"].apply(lambda t: len(HOSTILE_RX.findall(t)))
    table["coop_hits"] = table["clean"].apply(lambda t: len(COOP_RX.findall(t)))
    return table


def aggregate_by_year(table, sentence_table=None):
    """Add up per country–year and compute the metrics (per 1,000 words)."""
    g = (table.groupby(["country", "year"])
              .agg(documents=("doc_id", "count"), words=("words", "sum"),
                   hostile_hits=("hostile_hits", "sum"), coop_hits=("coop_hits", "sum"))
              .reset_index())
    g["hostility"] = g.hostile_hits / g.words * 1000
    g["cooperation"] = g.coop_hits / g.words * 1000
    total = (g.coop_hits + g.hostile_hits).replace(0, np.nan)
    g["tone_balance"] = (g.coop_hits - g.hostile_hits) / total
    if sentence_table is not None:
        v = (sentence_table.groupby(["country", "year"])
                .agg(vader_mean=("vader", "mean"), negative_share=("is_negative", "mean"))
                .reset_index())
        g = g.merge(v, on=["country", "year"], how="left")
    return g


docs = count_dictionary_terms(docs)
results = aggregate_by_year(docs, sentences)

# Average of all countries (each country weighted equally)
world_average = None
if COMPARE_TO_AVERAGE and world_docs.country.nunique() > len(COUNTRIES):
    print("⏳ Computing the all-country average (about 1–2 minutes)...")
    world = aggregate_by_year(count_dictionary_terms(world_docs))
    world_average = world.groupby("year")[["hostility", "cooperation", "tone_balance"]].mean().reset_index()

print("📊 Results by country–year (first 5 rows):")
results.head()

### 5.1 Validation: *which words* drive the scores?

Many people skip this step, but it **matters a lot academically**. If a country's hostility score is high mainly because of the word *"terrorism"* (because it condemns terrorism), that is very different from using many words like *"aggression"* or *"hegemony"* aimed at another country. Look at this list before interpreting the charts.

In [ ]:
def top_terms(code, regex, n=12):
    text = " ".join(docs.loc[docs.country == code, "clean"])
    counts = Counter(re.sub(r"\s+", " ", m.lower()) for m in regex.findall(text))
    return ", ".join(f"{term} ({count})" for term, count in counts.most_common(n))


for code in COUNTRIES:
    if (docs.country == code).any():
        print(f"🔴 {code} – most used HOSTILE words: {top_terms(code, HOSTILE_RX)}")
        print(f"🟢 {code} – most used COOPERATIVE words: {top_terms(code, COOP_RX)}\n")

## Step 6 – Track keywords over the years

Besides "tone", **what a country talks about more** is also a political signal: e.g. how often Russia mentions "NATO" or "multipolar", or China mentions "hegemon*". The table below counts the keywords from Step 1, per 1,000 words.

In [ ]:
keyword_counts = docs[["country", "year", "words"]].copy()
for kw in KEYWORDS:
    rx = build_regex([kw])
    keyword_counts[kw] = docs["clean"].apply(lambda t: len(rx.findall(t)))

keywords_by_year = keyword_counts.groupby(["country", "year"]).sum(numeric_only=True).reset_index()
for kw in KEYWORDS:
    keywords_by_year[kw] = keywords_by_year[kw] / keywords_by_year["words"] * 1000

print("Total mentions of each keyword (whole period):")
keyword_counts.groupby("country")[KEYWORDS].sum()

## Step 7 – Visualisation

Three kinds of charts:
1. **Trend lines** for each metric – one fixed colour per country, a dashed grey line for the all-country average, dotted vertical lines for events.
2. **A keyword heatmap** for each country (darker = mentioned more often).
3. **One keyword across countries.**

All charts are saved in the `charts/` folder so you can use them in your report.

In [ ]:
# Fixed palette: each country keeps the same colour in every chart
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
TEXT, TEXT_MUTED, GRID, BACKGROUND, AVERAGE_GREY = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb", "#8a8984"
if len(COUNTRIES) > len(PALETTE):
    raise ValueError("At most 8 countries per chart – remove some in Step 1.")
COLORS = {code: PALETTE[i] for i, code in enumerate(COUNTRIES)}

COUNTRY_NAMES = {"RUS": "Russia", "USA": "United States", "CHN": "China", "VNM": "Vietnam",
                 "GBR": "United Kingdom", "FRA": "France", "DEU": "Germany", "JPN": "Japan", "IND": "India",
                 "UKR": "Ukraine", "PRK": "North Korea", "KOR": "South Korea", "IRN": "Iran", "ISR": "Israel",
                 "PSE": "Palestine", "TUR": "Türkiye", "BRA": "Brazil", "ZAF": "South Africa",
                 "PHL": "Philippines", "AUS": "Australia"}
name = lambda code: COUNTRY_NAMES.get(code, code)

plt.rcParams.update({
    "figure.facecolor": BACKGROUND, "axes.facecolor": BACKGROUND, "savefig.facecolor": BACKGROUND,
    "axes.edgecolor": GRID, "axes.labelcolor": TEXT_MUTED, "text.color": TEXT,
    "xtick.color": TEXT_MUTED, "ytick.color": TEXT_MUTED, "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
})
Path("charts").mkdir(exist_ok=True)


def _spread_labels(last_y, gap):
    """Push end-of-line labels apart so they don't overlap."""
    positions, previous = {}, None
    for code in sorted(last_y, key=last_y.get):
        y = last_y[code] if previous is None else max(last_y[code], previous + gap)
        positions[code], previous = y, y
    return positions


def plot_trend(table, col, title, y_label, file_name, average=None, note=None):
    """Line chart of one metric for the chosen countries."""
    fig, ax = plt.subplots(figsize=(11, 5.6))
    last = {}
    for code in COUNTRIES:
        d = table[table.country == code].sort_values("year")
        if d.empty:
            continue
        ax.plot(d.year, d[col], color=COLORS[code], lw=2, marker="o", ms=4.5,
                markeredgecolor=BACKGROUND, markeredgewidth=1, label=name(code), zorder=3)
        last[code] = (d.year.iloc[-1], d[col].iloc[-1])
    if average is not None and col in average:
        ax.plot(average.year, average[col], color=AVERAGE_GREY, lw=1.6, ls="--",
                label="All-country average", zorder=2)

    # Direct labels at the end of each line
    y_min, y_max = ax.get_ylim()
    positions = _spread_labels({c: v[1] for c, v in last.items()}, (y_max - y_min) * 0.055)
    for code, (x, y) in last.items():
        ax.scatter([x + 0.55], [positions[code]], s=30, color=COLORS[code], clip_on=False, zorder=4)
        ax.text(x + 0.85, positions[code], name(code), va="center", fontsize=10, color=TEXT, clip_on=False)
    ax.set_ylim(y_min, max(y_max, max(positions.values(), default=y_max)))

    # Events: dotted vertical line + label above the plot (two staggered rows so they don't overlap)
    first_year, last_year = table.year.min(), table.year.max()
    shown = sorted(y for y in EVENTS if first_year <= y <= last_year)
    for i, year in enumerate(shown):
        ax.axvline(year, color="#c3c2bd", lw=1, ls=":", zorder=1)
        ax.text(year, 1.015 + 0.06 * (i % 2), EVENTS[year], transform=ax.get_xaxis_transform(),
                ha="center", va="bottom", fontsize=8.5, color=TEXT_MUTED, clip_on=False)

    ax.set_xlim(first_year - 0.5, last_year + 3.2)
    ax.grid(axis="y", color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.set_ylabel(y_label)
    ax.set_title(title, loc="left", fontsize=14, fontweight="bold", pad=40 if shown else 14)
    if note:
        fig.text(0.01, 0.005, note, fontsize=8.5, color=TEXT_MUTED)
    ax.legend(loc="upper left", bbox_to_anchor=(0, -0.12), ncol=5, frameon=False, fontsize=9.5)
    fig.tight_layout()
    fig.savefig(f"charts/{file_name}.png", dpi=160, bbox_inches="tight")
    plt.show()


source_note = ("Source: UN General Debate Corpus (Jankin, Baturo & Dasandi 2024)"
               if DATA_SOURCE == "UNGDC" else "Source: own data collection")

plot_trend(results, "hostility", "Hostile language over time",
           "Hostile words / 1,000 words", "1_hostility", world_average, source_note)
plot_trend(results, "cooperation", "Cooperative language over time",
           "Cooperative words / 1,000 words", "2_cooperation", world_average, source_note)
plot_trend(results, "tone_balance", "Tone balance: confrontational (−1) ↔ conciliatory (+1)",
           "Tone balance", "3_tone_balance", world_average, source_note)
plot_trend(results, "vader_mean", "Average sentence sentiment (VADER)",
           "Mean compound score", "4_vader", None, source_note)

In [ ]:
def plot_keyword_heatmap(code):
    """Heatmap: rows = years, columns = keywords, darker = mentioned more often (per 1,000 words)."""
    d = keywords_by_year[keywords_by_year.country == code].set_index("year")[KEYWORDS]
    if d.empty:
        print(f"No data for {code}")
        return
    blues = LinearSegmentedColormap.from_list(
        "blues", ["#f3f7fd", "#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
    fig, ax = plt.subplots(figsize=(1.0 * len(KEYWORDS) + 2.5, 0.32 * len(d) + 2))
    mesh = ax.pcolormesh(d.values, cmap=blues, vmin=0, edgecolors=BACKGROUND, linewidth=1.5)
    ax.invert_yaxis()
    ax.set_xticks(np.arange(len(KEYWORDS)) + 0.5, [k.replace("*", "…") for k in KEYWORDS],
                  rotation=40, ha="right")
    ax.set_yticks(np.arange(len(d)) + 0.5, d.index)
    ax.tick_params(length=0)
    ax.set_title(f"{name(code)}: keyword frequency by year (per 1,000 words)",
                 loc="left", fontsize=13, fontweight="bold", pad=12)
    for s in ax.spines.values():
        s.set_visible(False)
    bar = fig.colorbar(mesh, ax=ax, shrink=0.6)
    bar.outline.set_visible(False)
    fig.tight_layout()
    fig.savefig(f"charts/5_keywords_{code}.png", dpi=160, bbox_inches="tight")
    plt.show()


for code in COUNTRIES:
    plot_keyword_heatmap(code)

In [ ]:
# ✏️ Change the keyword here to compare it across countries (it must be in KEYWORDS)
COMPARE_KEYWORD = "sovereign*"

if COMPARE_KEYWORD not in KEYWORDS:
    raise ValueError(f"'{COMPARE_KEYWORD}' is not in KEYWORDS in Step 1.")
plot_trend(keywords_by_year, COMPARE_KEYWORD,
           f"How often “{COMPARE_KEYWORD.replace('*', '…')}” is mentioned over time",
           "Mentions / 1,000 words", "6_keyword_comparison", None, source_note)

## Step 8 – Find turning points & close-read to validate

Numbers are only **signals**; the real evidence is in the words. The standard "text as data" workflow:
1. Use the numbers to **find** the years when rhetoric changed most.
2. **Close-read** the most "heated" sentences of that year to understand *why*.
3. Quote those sentences as qualitative evidence in your paper.

In [ ]:
def find_shifts(col="tone_balance", n=8):
    """List the largest changes compared with the previous year."""
    b = results.sort_values(["country", "year"]).copy()
    b["from_year"] = b.groupby("country")["year"].shift()
    b["change"] = b.groupby("country")[col].diff()
    b = b.dropna(subset=["change"])
    b = b.reindex(b["change"].abs().sort_values(ascending=False).index).head(n)
    b["direction"] = np.where(b["change"] < 0, "⬇️ more confrontational", "⬆️ more conciliatory")
    b["from_year"] = b["from_year"].astype(int)
    return b[["country", "from_year", "year", col, "change", "direction"]].round(3)


find_shifts("tone_balance")

In [ ]:
sentences["hostile_hits"] = sentences["sentence"].apply(lambda s: len(HOSTILE_RX.findall(s)))
sentences["coop_hits"] = sentences["sentence"].apply(lambda s: len(COOP_RX.findall(s)))


def highlight(sentence):
    """Put hostile dictionary words in CAPITALS so they stand out."""
    return HOSTILE_RX.sub(lambda m: m.group(0).upper(), sentence)


def close_read(code, year, n=5):
    """Print the most notable sentences of one country's speech in one year."""
    s = sentences[(sentences.country == code) & (sentences.year == year)].drop_duplicates("sentence")
    if s.empty:
        print(f"No data for {code} in {year}")
        return
    print(f"═══ {name(code)} – {year}: {len(s)} sentences ═══\n")
    print(f"🔴 {n} sentences with the most hostile words (in CAPITALS):")
    for _, r in s.sort_values(["hostile_hits", "vader"], ascending=[False, True]).head(n).iterrows():
        if r.hostile_hits > 0:
            print(f"  • [{r.hostile_hits} words] {highlight(r.sentence)}\n")
    print(f"🟠 {n} sentences VADER scored most negative:")
    for _, r in s.nsmallest(n, "vader").iterrows():
        print(f"  • [{r.vader:+.2f}] {r.sentence}\n")
    print(f"🟢 {n} most cooperative sentences:")
    for _, r in s.nlargest(n, "coop_hits").iterrows():
        if r.coop_hits > 0:
            print(f"  • [{r.coop_hits} words] {r.sentence}\n")


# ✏️ Change the country code and year to read, e.g. close_read("RUS", 2022).
#    Tip: take a row from the "turning points" table above. Default: first country, latest year.
close_read(COUNTRIES[0], int(results.year.max()))

## Step 9 – *(Advanced, optional)* Classify tone with a zero-shot AI model

VADER and the dictionaries only **count words**, so they miss context: the sentence *"We do not seek confrontation"* still counts as hostile. A **zero-shot** model (here `facebook/bart-large-mnli`) **reads the whole sentence** and decides which of the labels we give it fits best – no extra training needed.

**How to turn it on:**
1. In Colab: *Runtime → Change runtime type → choose **T4 GPU*** → Save. (This restarts the machine; run the cells from the top again.)
2. Set `RUN_ADVANCED = True` and run the cell below. The first run takes a few minutes to download the model (~1.6 GB).

To save time, we only take `SENTENCES_PER_TEXT` random sentences from each text.

In [ ]:
RUN_ADVANCED = False      # ✏️ set to True to run
SENTENCES_PER_TEXT = 30

if RUN_ADVANCED:
    import torch
    from transformers import pipeline

    device = 0 if torch.cuda.is_available() else -1
    if device == -1:
        print("⚠️ No GPU found – this will be very slow. Turn on the T4 GPU as described above.")
    classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=device)

    LABELS = {"hostile or confrontational": "AI: hostile",
              "cooperative or conciliatory": "AI: cooperative",
              "neutral or procedural": "AI: neutral"}
    sample = (sentences.sample(frac=1, random_state=42)
                       .groupby("doc_id").head(SENTENCES_PER_TEXT).copy())
    print(f"🤖 Classifying {len(sample):,} sentences...")
    predictions = classifier(sample["sentence"].tolist(), candidate_labels=list(LABELS),
                             hypothesis_template="The tone of this statement is {}.", batch_size=16)
    sample["ai_label"] = [LABELS[p["labels"][0]] for p in predictions]

    ai_by_year = (pd.crosstab([sample.country, sample.year], sample.ai_label, normalize="index")
                    .reset_index())
    ai_by_year.columns.name = None
    if "AI: hostile" in ai_by_year:
        plot_trend(ai_by_year, "AI: hostile", "Share of sentences the AI rated hostile",
                   "Share of sentences (0–1)", "7_ai_hostile_share", None, source_note)
    print("Examples of sentences the AI rated hostile:")
    display(sample[sample.ai_label == "AI: hostile"][["country", "year", "sentence"]].head(8))

## Step 10 – Export the results

The cell below saves:
- `results_by_year.csv` – every metric per country–year (opens in Excel),
- `keywords_by_year.csv` – keyword frequencies,
- `validation_sample.csv` – **60 random sentences** with the machine's scores and an empty `human_label` column for **you to label by hand** (hostile / cooperative / neutral). Comparing your labels with the machine's gives the *reliability* of the method – a very valuable paragraph for your methods section,
- the `charts/` folder with every chart as PNG.

Everything is zipped into `project_results.zip` and downloaded to your computer.

In [ ]:
results.round(4).to_csv("results_by_year.csv", index=False, encoding="utf-8-sig")
keywords_by_year.round(4).to_csv("keywords_by_year.csv", index=False, encoding="utf-8-sig")

validation = sentences.sample(min(60, len(sentences)), random_state=7)[
    ["country", "year", "sentence", "vader", "hostile_hits", "coop_hits"]].copy()
validation["human_label"] = ""
validation.to_csv("validation_sample.csv", index=False, encoding="utf-8-sig")

with zipfile.ZipFile("project_results.zip", "w") as z:
    for f in ["results_by_year.csv", "keywords_by_year.csv", "validation_sample.csv",
              *glob.glob("charts/*.png")]:
        z.write(f)
print("💾 Saved project_results.zip")

try:
    from google.colab import files
    files.download("project_results.zip")
except ImportError:
    pass

## 📝 Interpreting the results & method limitations *(read before writing your report)*

**Questions to ask when reading the charts**
- In which year is each country's biggest turning point? Does it coincide with an event? What does the speech of that year *say* (use `close_read`)?
- Does a rising hostility score mean the country is **attacking** others, or **describing threats** it faces? (See Step 5.1.)
- Which countries are **stable** in their rhetoric, and which **fluctuate** strongly? What does that say about their foreign-communication strategy?

**Limitations to state in your paper**
1. **Tone ≠ policy.** Language at the UN is deliberate performance (*signalling*); it does not directly reflect behaviour.
2. **Word counting ignores context** – negation, irony and quoting others are all counted the same way.
3. **Translation effects** – UNGDC texts are English versions; part of the tone comes from the translator.
4. **Changing speakers** (head of state, prime minister, foreign minister or ambassador) can change the style. UNGDC includes speaker data so you can control for this.
5. **Always validate by hand** with `validation_sample.csv`.

**Recommended methods reading:** Grimmer, J. & Stewart, B. M. (2013). *Text as Data: The Promise and Pitfalls of Automatic Content Analysis Methods for Political Texts.* Political Analysis, 21(3), 267–297.

**Possible extension:** compare the results with **UN General Assembly voting data** (Voeten, Harvard Dataverse) to see whether a country's "words" and "votes" move together.